# Unfair Clause Classification: All Experiments

## Setup

In [1]:
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM,
                          TrainingArguments, Trainer)

In [2]:
DATA_DIR = "/kaggle/input/datasets/mohmmedtamer/sic-graduation-project-bert"
WORK_DIR = Path("/kaggle/working")

LEGAL_BERT = "nlpaueb/legal-bert-base-uncased"
MULTI_MODEL = "xlm-roberta-base"
TRANSLATOR = "facebook/nllb-200-distilled-600M"

NAMES = ["Limitation of liability", "Unilateral termination", "Unilateral change", "Content removal",
         "Contract by using", "Choice of law", "Jurisdiction", "Arbitration"]

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## Load data

In [3]:
def to_vector(labels):
    ids = [int(x) for x in re.findall(r"\d+", str(labels))]
    return [1.0 if i in ids else 0.0 for i in range(8)]

def load(path):
    df = pd.read_csv(path)
    df["text"] = df["text"].fillna("").astype(str)
    df["y"] = df["labels"].apply(to_vector)
    return df

def labels_of(df):
    return np.array(df["y"].tolist())

train = load(f"{DATA_DIR}/train.csv")
val = load(f"{DATA_DIR}/val.csv")
test = load(f"{DATA_DIR}/test.csv")

y_train, y_val, y_test = labels_of(train), labels_of(val), labels_of(test)

len(train), len(val), len(test)

(5532, 2275, 1607)

## Helpers

In [4]:
def macro(y, pred):
    return round(f1_score(y, pred, average="macro", zero_division=0) * 100, 1)

def micro(y, pred):
    return round(f1_score(y, pred, average="micro", zero_division=0) * 100, 1)

def classify(model, tokenizer, texts, batch_size=64):
    model.eval()
    probs = []
    for i in range(0, len(texts), batch_size):
        inputs = tokenizer(texts[i:i + batch_size], truncation=True, max_length=128, padding=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            probs.append(torch.sigmoid(model(**inputs).logits).float().cpu().numpy())
    return np.vstack(probs)

def single_threshold(y, probs):
    best_t, best_f1 = 0.5, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(y, probs > t, average="macro", zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    return np.full(8, round(best_t, 2))

def per_label_thresholds(y, probs):
    thresholds = []
    for i in range(8):
        best_t, best_f1 = 0.5, 0
        for t in np.arange(0.1, 0.9, 0.05):
            f1 = f1_score(y[:, i], probs[:, i] > t, zero_division=0)
            if f1 > best_f1:
                best_t, best_f1 = t, f1
        thresholds.append(round(best_t, 2))
    return np.array(thresholds)

def train_model(model_name, train_df, val_df, epochs, keep_best, name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128, padding="max_length")

    def to_ds(df):
        return Dataset.from_dict({"text": df["text"].tolist(), "labels": df["y"].tolist()}).map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, problem_type="multi_label_classification")

    steps = epochs * int(np.ceil(len(train_df) / (16 * max(torch.cuda.device_count(), 1))))
    args = TrainingArguments(
        output_dir=str(WORK_DIR / f"checkpoints_{name}"),
        num_train_epochs=epochs,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        warmup_steps=int(0.1 * steps),
        eval_strategy="epoch",
        save_strategy="epoch" if keep_best else "no",
        save_total_limit=1,
        load_best_model_at_end=keep_best,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    Trainer(model=model, args=args, train_dataset=to_ds(train_df), eval_dataset=to_ds(val_df)).train()
    shutil.rmtree(WORK_DIR / f"checkpoints_{name}", ignore_errors=True)
    return model, tokenizer

def save_model(model, tokenizer, thresholds, name):
    path = WORK_DIR / "models" / name
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    (path / "thresholds.txt").write_text(",".join(str(t) for t in thresholds))

results = []

## Experiment 1: TF-IDF baseline

In [5]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
baseline = OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight="balanced"))
baseline.fit(tfidf.fit_transform(train["text"]), y_train)

pred = baseline.predict(tfidf.transform(test["text"]))
results.append({"Experiment": "1. TF-IDF baseline", "EN micro F1": micro(y_test, pred), "EN macro F1": macro(y_test, pred)})
results[-1]

{'Experiment': '1. TF-IDF baseline', 'EN micro F1': 67.7, 'EN macro F1': 69.0}

## Experiment 2: Legal-BERT, 5 epochs, one threshold

In [6]:
model_2, tokenizer_2 = train_model(LEGAL_BERT, train, val, epochs=5, keep_best=False, name="exp2")

thresholds_2 = single_threshold(y_val, classify(model_2, tokenizer_2, val["text"].tolist()))
pred = classify(model_2, tokenizer_2, test["text"].tolist()) > thresholds_2

results.append({"Experiment": "2. Legal-BERT (5 ep, one threshold)", "EN micro F1": micro(y_test, pred), "EN macro F1": macro(y_test, pred)})
print("Threshold:", thresholds_2[0])
results[-1]

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,0.145846
2,No log,0.081559
3,0.248913,0.061000
4,0.248913,0.056114
5,0.248913,0.054856


Threshold: 0.5


{'Experiment': '2. Legal-BERT (5 ep, one threshold)',
 'EN micro F1': 81.7,
 'EN macro F1': 84.3}

In [7]:
del model_2
torch.cuda.empty_cache()

## Experiment 3: Legal-BERT, 10 epochs, best epoch, per-label thresholds

In [8]:
model_3, tokenizer_3 = train_model(LEGAL_BERT, train, val, epochs=10, keep_best=True, name="exp3")

thresholds_3 = per_label_thresholds(y_val, classify(model_3, tokenizer_3, val["text"].tolist()))
pred = classify(model_3, tokenizer_3, test["text"].tolist()) > thresholds_3

results.append({"Experiment": "3. Legal-BERT (10 ep, per-label thresholds)", "EN micro F1": micro(y_test, pred), "EN macro F1": macro(y_test, pred)})
save_model(model_3, tokenizer_3, thresholds_3, "legal-bert")
pd.Series(thresholds_3, index=NAMES)

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Limitation of liability    0.65
Unilateral termination     0.80
Unilateral change          0.60
Content removal            0.65
Contract by using          0.85
Choice of law              0.15
Jurisdiction               0.60
Arbitration                0.60
dtype: float64

In [9]:
print(classification_report(y_test, pred, target_names=NAMES, zero_division=0))

                         precision    recall  f1-score   support

Limitation of liability       0.84      0.68      0.75        38
 Unilateral termination       0.81      0.68      0.74        38
      Unilateral change       0.76      0.66      0.70        38
        Content removal       0.85      0.85      0.85        13
      Contract by using       1.00      0.74      0.85        23
          Choice of law       0.86      0.92      0.89        13
           Jurisdiction       0.94      1.00      0.97        16
            Arbitration       0.75      0.86      0.80         7

              micro avg       0.84      0.75      0.79       186
              macro avg       0.85      0.80      0.82       186
           weighted avg       0.84      0.75      0.79       186
            samples avg       0.08      0.08      0.08       186



## Translator

In [10]:
mt_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR).to(device).eval()
if device == "cuda":
    mt_model = mt_model.half()

def translate(texts, src, tgt, batch_size=32):
    mt_tokenizer.src_lang = src
    order = np.argsort([len(t) for t in texts])
    out = [""] * len(texts)
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        batch = [texts[j] for j in idx]
        inputs = mt_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            tokens = mt_model.generate(**inputs, forced_bos_token_id=mt_tokenizer.convert_tokens_to_ids(tgt),
                                       max_length=256, num_beams=2)
        for j, text in zip(idx, mt_tokenizer.batch_decode(tokens, skip_special_tokens=True)):
            out[j] = text
    return out

translate(["We may terminate your account at any time without notice."], "eng_Latn", "arb_Arab")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

['قد ننهي حسابك في أي وقت دون إشعار']

## Translate dataset to Arabic

In [11]:
AR_DIR = WORK_DIR / "arabic"
AR_DIR.mkdir(parents=True, exist_ok=True)

ar = {}
for name, df in [("train", train), ("val", val), ("test", test)]:
    path = AR_DIR / f"{name}.csv"
    if not path.exists():
        start = time.time()
        arabic = translate(df["text"].tolist(), "eng_Latn", "arb_Arab")
        pd.DataFrame({"text": arabic, "labels": df["labels"]}).to_csv(path, index=False)
        print(name, "translated in", round(time.time() - start), "sec")
    ar[name] = load(path)

y_val_ar, y_test_ar = labels_of(ar["val"]), labels_of(ar["test"])

pd.DataFrame({"English": test["text"].head(5), "Arabic": ar["test"]["text"].head(5)})

,English,Arabic
0,"last updated date : may 15 , 2017 \n",آخر تاريخ محدد: 15 أيار / مايو 2017
1,"academia , inc. ( `` academia.edu '' or `` we ...","academemia , inc. ( `` academia.edu ' أو `` we..."
2,please read carefully the following terms and ...,"يرجى قراءة الشروط والأحكام التالية ( `` شروط ""..."
3,these terms govern your access to and use of t...,هذه الشروط تحكم وصولك إلى الموقع والخدمات والم...
4,arbitration notice : unless you opt out of arb...,إشعار التحكيم: ما لم تقم بالتخلي عن التحكيم في...


## Experiment 4: Arabic via translation + Legal-BERT

In [12]:
def predict_4(texts):
    return classify(model_3, tokenizer_3, translate(texts, "arb_Arab", "eng_Latn")) > thresholds_3

en_pred = classify(model_3, tokenizer_3, test["text"].tolist()) > thresholds_3

start = time.time()
ar_pred = predict_4(ar["test"]["text"].tolist())
seconds = time.time() - start

results.append({
    "Experiment": "4. AR: translate + Legal-BERT",
    "EN micro F1": micro(y_test, en_pred), "EN macro F1": macro(y_test, en_pred),
    "AR micro F1": micro(y_test_ar, ar_pred), "AR macro F1": macro(y_test_ar, ar_pred),
    "Sec / 100 AR clauses": round(seconds / len(ar_pred) * 100, 2),
})
results[-1]

{'Experiment': '4. AR: translate + Legal-BERT',
 'EN micro F1': 79.2,
 'EN macro F1': 81.9,
 'AR micro F1': 77.9,
 'AR macro F1': 80.3,
 'Sec / 100 AR clauses': 5.64}

## Experiment 5: XLM-R trained on English + Arabic

In [13]:
train_mix = pd.concat([train, ar["train"]], ignore_index=True)
val_mix = pd.concat([val, ar["val"]], ignore_index=True)

model_5, tokenizer_5 = train_model(MULTI_MODEL, train_mix, val_mix, epochs=5, keep_best=True, name="exp5")
thresholds_5 = per_label_thresholds(labels_of(val_mix), classify(model_5, tokenizer_5, val_mix["text"].tolist()))

def predict_5(texts):
    return classify(model_5, tokenizer_5, texts) > thresholds_5

en_pred = predict_5(test["text"].tolist())

start = time.time()
ar_pred = predict_5(ar["test"]["text"].tolist())
seconds = time.time() - start

results.append({
    "Experiment": "5. XLM-R (EN + AR)",
    "EN micro F1": micro(y_test, en_pred), "EN macro F1": macro(y_test, en_pred),
    "AR micro F1": micro(y_test_ar, ar_pred), "AR macro F1": macro(y_test_ar, ar_pred),
    "Sec / 100 AR clauses": round(seconds / len(ar_pred) * 100, 2),
})
save_model(model_5, tokenizer_5, thresholds_5, "xlmr-en-ar")
results[-1]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/11064 [00:00<?, ? examples/s]

Map:   0%|          | 0/4550 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'Experiment': '5. XLM-R (EN + AR)',
 'EN micro F1': 62.3,
 'EN macro F1': 61.6,
 'AR micro F1': 61.4,
 'AR macro F1': 59.2,
 'Sec / 100 AR clauses': 0.64}

## Experiment 6: XLM-R trained on English only

In [14]:
model_6, tokenizer_6 = train_model(MULTI_MODEL, train, val, epochs=10, keep_best=True, name="exp6")
thresholds_6 = per_label_thresholds(y_val, classify(model_6, tokenizer_6, val["text"].tolist()))

def predict_6(texts):
    return classify(model_6, tokenizer_6, texts) > thresholds_6

en_pred = predict_6(test["text"].tolist())

start = time.time()
ar_pred = predict_6(ar["test"]["text"].tolist())
seconds = time.time() - start

results.append({
    "Experiment": "6. XLM-R (EN only)",
    "EN micro F1": micro(y_test, en_pred), "EN macro F1": macro(y_test, en_pred),
    "AR micro F1": micro(y_test_ar, ar_pred), "AR macro F1": macro(y_test_ar, ar_pred),
    "Sec / 100 AR clauses": round(seconds / len(ar_pred) * 100, 2),
})
save_model(model_6, tokenizer_6, thresholds_6, "xlmr-en")
results[-1]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'Experiment': '6. XLM-R (EN only)',
 'EN micro F1': 61.8,
 'EN macro F1': 61.1,
 'AR micro F1': 17.1,
 'AR macro F1': 9.0,
 'Sec / 100 AR clauses': 0.66}

## Results

In [22]:
pd.DataFrame(results).set_index("Experiment")

,EN micro F1,EN macro F1,AR micro F1,AR macro F1,Sec / 100 AR clauses
Experiment,,,,,
1. TF-IDF baseline,67.7,69.0,NaN,NaN,NaN
"2. Legal-BERT (5 ep, one threshold)",81.7,84.3,NaN,NaN,NaN
"3. Legal-BERT (10 ep, per-label thresholds)",79.2,81.9,NaN,NaN,NaN
4. AR: translate + Legal-BERT,79.2,81.9,77.9,80.3,5.64
5. XLM-R (EN + AR),62.3,61.6,61.4,59.2,0.64
6. XLM-R (EN only),61.8,61.1,17.1,9.0,0.66


In [23]:
def model_size(model):
    return round(sum(p.numel() for p in model.parameters()) / 1e6)

pd.DataFrame({
    "Models at inference": ["NLLB + Legal-BERT", "XLM-R", "XLM-R"],
    "Parameters (M)": [model_size(mt_model) + model_size(model_3), model_size(model_5), model_size(model_6)],
}, index=["4", "5", "6"])

,Models at inference,Parameters (M)
4,NLLB + Legal-BERT,724
5,XLM-R,278
6,XLM-R,278


## Arabic examples

In [17]:
arabic_predictors = {"4": predict_4, "5": predict_5, "6": predict_6}

arabic_examples = [
    "يحق لنا إنهاء حسابك في أي وقت ودون إشعار مسبق.",
    "يتم حل أي نزاع ينشأ عن هذه الشروط عن طريق التحكيم الملزم.",
    "تخضع هذه الشروط لقوانين ولاية كاليفورنيا.",
    "لا نتحمل أي مسؤولية عن أي أضرار تنتج عن استخدامك للخدمة.",
    "يمكنك التواصل مع فريق الدعم عبر البريد الإلكتروني.",
]

table = {"Clause": arabic_examples}
for name, predict in arabic_predictors.items():
    table[name] = [[NAMES[i] for i in np.where(row)[0]] for row in predict(arabic_examples)]

pd.DataFrame(table)

,Clause,4,5,6
0,يحق لنا إنهاء حسابك في أي وقت ودون إشعار مسبق.,[Unilateral termination],"[Unilateral termination, Content removal]",[]
1,يتم حل أي نزاع ينشأ عن هذه الشروط عن طريق التح...,[Arbitration],[],[]
2,تخضع هذه الشروط لقوانين ولاية كاليفورنيا.,[Choice of law],"[Jurisdiction, Arbitration]",[]
3,لا نتحمل أي مسؤولية عن أي أضرار تنتج عن استخدا...,[],[Limitation of liability],[]
4,يمكنك التواصل مع فريق الدعم عبر البريد الإلكتر...,[],[],[]


## Manual Arabic test (optional)

In [18]:
MANUAL_TEST = f"{DATA_DIR}/manual_ar.csv"

if Path(MANUAL_TEST).exists():
    manual = load(MANUAL_TEST)
    y_manual = labels_of(manual)
    rows = []
    for name, predict in arabic_predictors.items():
        pred = predict(manual["text"].tolist())
        rows.append({"Experiment": name, "macro F1": macro(y_manual, pred), "micro F1": micro(y_manual, pred)})
    display(pd.DataFrame(rows).set_index("Experiment"))
else:
    print("No manual test file found")

No manual test file found


In [19]:
sorted(str(p.relative_to(WORK_DIR)) for p in (WORK_DIR / "models").rglob("thresholds.txt"))

['models/legal-bert/thresholds.txt',
 'models/xlmr-en-ar/thresholds.txt',
 'models/xlmr-en/thresholds.txt']